In [195]:
import time

import xarray as xr
from multiformats import CID
from py_hamt import HAMT, KuboCAS, ZarrHAMTStore
from xarray import Dataset

ipns_hash = "bafyr4iecw3faqyvj75psutabk2jxpddpjdokdy5b26jdnjjzpkzbgb5xoq"

# Create a content address store instance using the IPFS gateway URL
kubo_cas = KuboCAS(gateway_base_url="http://127.0.0.1:8080")

# Decode the root CID for the Zarr dataset
root_node_id = CID.decode(ipns_hash)

# Create HAMT instance using the IPFSStore
hamt = HAMT(cas=kubo_cas, values_are_bytes=True, root_node_id=root_node_id, read_only=True)

# Initialize the store
zhs = ZarrHAMTStore(hamt, read_only=True)

# Open the dataset with xarray
zarr_ds: Dataset = xr.open_zarr(store=zhs, zarr_format=3)

# Sort the dataset by latitude and longitude
zarr_ds = zarr_ds.sortby("longitude")
zarr_ds = zarr_ds.sortby("latitude")

print(zarr_ds)

<xarray.Dataset> Size: 967MB
Dimensions:    (time: 6717, latitude: 120, longitude: 300)
Coordinates:
  * time       (time) datetime64[ns] 54kB 2007-01-01 2007-01-02 ... 2025-05-22
  * longitude  (longitude) float32 1kB -129.9 -129.6 -129.4 ... -55.38 -55.12
  * latitude   (latitude) float32 480B 20.12 20.38 20.62 ... 49.38 49.62 49.88
Data variables:
    precip     (time, latitude, longitude) float32 967MB ...
Attributes:
    title:          CPC Unified Gauge-Based Analysis of Daily Precipitation o...
    Conventions:    COARDS
    history:        created 04/2010 by CAS from data obtained from NCEP/CPC\n...
    description:    Gridded daily Precipitation
    platform:       Observations
    Comments:       Preciptation is accumulated from 12z of previous day to 1...
    dataset_title:  CPC Unified Gauge-Based Analysis of Daily Precipitation o...
    References:     http://www.psl.noaa.gov/data/gridded/data.unified.daily.c...


In [196]:
lat_bounds = (20, 30)
lon_bounds = (-110, -70)
time_bounds = ("2007-01-01", "2025-12-31")  # Matching available data

sliced_ds = zarr_ds.sel(
    latitude=slice(*lat_bounds),
    longitude=slice(*lon_bounds),
    time=slice(*time_bounds)
)
print(sliced_ds)

<xarray.Dataset> Size: 172MB
Dimensions:    (time: 6717, latitude: 40, longitude: 160)
Coordinates:
  * time       (time) datetime64[ns] 54kB 2007-01-01 2007-01-02 ... 2025-05-22
  * longitude  (longitude) float32 640B -109.9 -109.6 -109.4 ... -70.38 -70.12
  * latitude   (latitude) float32 160B 20.12 20.38 20.62 ... 29.38 29.62 29.88
Data variables:
    precip     (time, latitude, longitude) float32 172MB ...
Attributes:
    title:          CPC Unified Gauge-Based Analysis of Daily Precipitation o...
    Conventions:    COARDS
    history:        created 04/2010 by CAS from data obtained from NCEP/CPC\n...
    description:    Gridded daily Precipitation
    platform:       Observations
    Comments:       Preciptation is accumulated from 12z of previous day to 1...
    dataset_title:  CPC Unified Gauge-Based Analysis of Daily Precipitation o...
    References:     http://www.psl.noaa.gov/data/gridded/data.unified.daily.c...


In [197]:
# Trigger computation / download
start_time = time.time()
sliced_ds.load()
end_time = time.time()

download_time = end_time - start_time
download_size_mb = sliced_ds.nbytes / (1024 * 1024)
download_speed_mbs = download_size_mb / download_time if download_time > 0 else 0.0

print(f"Download and subset completed in {download_time:.2f} seconds. Speed: {download_speed_mbs:.2f} MB/s")
print(sliced_ds)

Download and subset completed in 1.40 seconds. Speed: 117.17 MB/s
<xarray.Dataset> Size: 172MB
Dimensions:    (time: 6717, latitude: 40, longitude: 160)
Coordinates:
  * time       (time) datetime64[ns] 54kB 2007-01-01 2007-01-02 ... 2025-05-22
  * longitude  (longitude) float32 640B -109.9 -109.6 -109.4 ... -70.38 -70.12
  * latitude   (latitude) float32 160B 20.12 20.38 20.62 ... 29.38 29.62 29.88
Data variables:
    precip     (time, latitude, longitude) float32 172MB nan nan nan ... nan nan
Attributes:
    title:          CPC Unified Gauge-Based Analysis of Daily Precipitation o...
    Conventions:    COARDS
    history:        created 04/2010 by CAS from data obtained from NCEP/CPC\n...
    description:    Gridded daily Precipitation
    platform:       Observations
    Comments:       Preciptation is accumulated from 12z of previous day to 1...
    dataset_title:  CPC Unified Gauge-Based Analysis of Daily Precipitation o...
    References:     http://www.psl.noaa.gov/data/gridded

In [119]:
import time

import xarray as xr
from multiformats import CID
from py_hamt import HAMT, KuboCAS, ZarrHAMTStore
from xarray import Dataset

ipns_hash = "bafyr4iecw3faqyvj75psutabk2jxpddpjdokdy5b26jdnjjzpkzbgb5xoq"

# Create a content address store instance using the IPFS gateway URL
kubo_cas = KuboCAS(gateway_base_url="http://127.0.0.1:8080/")

# Decode the root CID for the Zarr dataset
root_node_id = CID.decode(ipns_hash)

# Create HAMT instance using the IPFSStore
hamt = HAMT(cas=kubo_cas, values_are_bytes=True, root_node_id=root_node_id, read_only=True)

# Initialize the store
zhs = ZarrHAMTStore(hamt, read_only=True)

# Open the dataset with xarray
zarr_ds: Dataset = xr.open_zarr(store=zhs, zarr_format=3)

# Sort the dataset by latitude and longitude
zarr_ds = zarr_ds.sortby("longitude")
zarr_ds = zarr_ds.sortby("latitude")
print(zarr_ds['precip'].encoding)
print(zarr_ds)

HTTPStatusError: Redirect response '301 Moved Permanently' for url 'http://127.0.0.1:8080//ipfs/bafyr4iecw3faqyvj75psutabk2jxpddpjdokdy5b26jdnjjzpkzbgb5xoq'
Redirect location: '/ipfs/bafyr4iecw3faqyvj75psutabk2jxpddpjdokdy5b26jdnjjzpkzbgb5xoq'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/301

In [ ]:

from numcodecs.zarr3 import PCodec, Delta, Quantize         # or `from zarr.codecs import PCodec`
# from numcodecs import Quantize
import shutil, time, humanize, zarr
from pathlib import Path
import numpy as np

pco = PCodec(level=8)  # or `PCodec(level=8, scale=0.5, dtype=np.float32)`
delta = Delta(dtype="float32")

encoding = {
    "precip": {                       # keep your chunk shape
        "chunks": (400, 25, 25),
        "filters": (Quantize(digits=3, dtype="float32"),), 
        "serializer": pco,            # <-- CORRECT SLOT
    }
}

da = zarr_ds["precip"]                            # an xarray.DataArray
dask_arr = da.data                                # pull out the dask array
dask_arr = dask_arr.map_blocks(
    np.ascontiguousarray, dtype=da.dtype)         # Dask API _accepts_ dtype
zarr_ds["precip"].data = dask_arr                 # put it back

# -------- 4.1  local DirectoryStore as a scratch pad
dst_dir = Path("cpc_precip_pcodec.zarr")
if dst_dir.exists():
    shutil.rmtree(dst_dir)          # start clean

t0 = time.time()
zarr_ds.to_zarr(
    store=dst_dir,
    mode="w",
    encoding=encoding,
    consolidated=True,
    zarr_format=3,
    compute=True,
)
print(f"PCodec write finished in {time.time()-t0:.1f}s")
# -------- 4.2  quick size check on disk
dir_bytes = sum(f.stat().st_size for f in dst_dir.rglob("*") if f.is_file())
print("Local size:", humanize.naturalsize(dir_bytes))

/Users/aristotle/Desktop/work/zarr-downloader-test/zarr_v3-hamt_v3/.venv/lib/python3.12/site-packages/numcodecs/zarr3.py:182: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/aristotle/Desktop/work/zarr-downloader-test/zarr_v3-hamt_v3/.venv/lib/python3.12/site-packages/numcodecs/zarr3.py:167: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


TypeError: Expected a ArrayArrayCodec. Got <class 'numcodecs.quantize.Quantize'> instead.

In [ ]:
async def load_zarr_to_ipfs(zarr_path: str, cid_out_path: str = None) -> str:
    """
    Put the Zarr store onto IPFS using py-hamt. Returns the root CID as a string.
    If cid_out_path is provided, also write the CID to a file.

    :param zarr_path: Path to the local Zarr directory.
    :param cid_out_path: Optional path to write the CID (e.g. "cpc_conus.cid").
    :return: The root CID string
    """
    print(f"Loading {zarr_path} onto IPFS via py-hamt...")

    kubo_cas = KuboCAS()
    hamt = await HAMT.build(cas=kubo_cas, values_are_bytes=True)
    zhs = ZarrHAMTStore(hamt)

    ds = xr.open_zarr(zarr_path)
    ds.to_zarr(store=zhs, mode="w")

    await hamt.make_read_only()
    root_cid_str = str(hamt.root_node_id)
    print(f"Successfully wrote data to IPFS. Root CID = {root_cid_str}")

    if cid_out_path:
        with open(cid_out_path, "w") as f:
            f.write(root_cid_str + "\n")

    return root_cid_str

In [6]:
start_time = time.time()
# 4. Put the new "cpc_conus_demo.zarr" on IPFS
root_cid = await load_zarr_to_ipfs(
    zarr_path=dst_dir,
    cid_out_path="cpc_conus_demo.cid"
)
end_time = time.time()

download_time = end_time - start_time

print(f"Download and subset completed in {download_time:.2f} seconds.")

print(f"Pipeline complete! The root CID is {root_cid}")

Loading cpc_precip_pcodec.zarr onto IPFS via py-hamt...


/Users/aristotle/Desktop/work/zarr-downloader-test/zarr_v3-hamt_v3/.venv/lib/python3.12/site-packages/numcodecs/zarr3.py:182: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/aristotle/Desktop/work/zarr-downloader-test/zarr_v3-hamt_v3/.venv/lib/python3.12/site-packages/numcodecs/zarr3.py:182: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/aristotle/Desktop/work/zarr-downloader-test/zarr_v3-hamt_v3/.venv/lib/python3.12/site-packages/zarr/api/asynchronous.py:213: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Successfully wrote data to IPFS. Root CID = bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la
Download and subset completed in 2.91 seconds.
Pipeline complete! The root CID is bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la


In [28]:
!ipfs dag stat bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la

CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 29756, NumBlocks: 1
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 261254, NumBlocks: 2
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 261362, NumBlocks: 3
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 523506, NumBlocks: 4
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 587911, NumBlocks: 5
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 587934, NumBlocks: 6
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 849677, NumBlocks: 7
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 849785, NumBlocks: 8
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 1111929, NumBlocks: 9
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Size: 1277266, NumBlocks: 10
CID: bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la, Si

In [26]:
ipns_hash = "bafyr4ib265x24nhtgmkxcgwzaxznhg6zj4zwpnzk3pv4fag4crat6hy3la"

# Create a content address store instance using the IPFS gateway URL
kubo_cas = KuboCAS(gateway_base_url="http://127.0.0.1:8080/")

# Decode the root CID for the Zarr dataset
root_node_id = CID.decode(ipns_hash)

# Create HAMT instance using the IPFSStore
hamt = HAMT(cas=kubo_cas, values_are_bytes=True, root_node_id=root_node_id, read_only=True)

# Initialize the store
zhs = ZarrHAMTStore(hamt, read_only=True)

# Open the dataset with xarray
zarr_ds: Dataset = xr.open_zarr(store=zhs, zarr_format=3)
print(zarr_ds)

print(zarr_ds['precip'].encoding)

<xarray.Dataset> Size: 967MB
Dimensions:    (time: 6717, longitude: 300, latitude: 120)
Coordinates:
  * time       (time) datetime64[ns] 54kB 2007-01-01 2007-01-02 ... 2025-05-22
  * longitude  (longitude) float32 1kB -129.9 -129.6 -129.4 ... -55.38 -55.12
  * latitude   (latitude) float32 480B 20.12 20.38 20.62 ... 49.38 49.62 49.88
Data variables:
    precip     (time, latitude, longitude) float32 967MB dask.array<chunksize=(400, 25, 25), meta=np.ndarray>
Attributes:
    title:          CPC Unified Gauge-Based Analysis of Daily Precipitation o...
    Conventions:    COARDS
    history:        created 04/2010 by CAS from data obtained from NCEP/CPC\n...
    description:    Gridded daily Precipitation
    platform:       Observations
    Comments:       Preciptation is accumulated from 12z of previous day to 1...
    dataset_title:  CPC Unified Gauge-Based Analysis of Daily Precipitation o...
    References:     http://www.psl.noaa.gov/data/gridded/data.unified.daily.c...
{'chunks': (

/Users/aristotle/Desktop/work/zarr-downloader-test/zarr_v3-hamt_v3/.venv/lib/python3.12/site-packages/numcodecs/zarr3.py:182: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


In [4]:
# 5 · Publish back to IPFS with HAMT (optional)
# initialise a fresh HAMT rooted at an empty CID
new_hamt = HAMT(cas=kubo_cas, values_are_bytes=True)
out_store = ZarrHAMTStore(new_hamt, read_only=False)

# copy chunks store → hamt in a streaming fashion
zarr.copy_store(zarr.DirectoryStore(dst_dir), out_store, if_exists="replace")

# flush & capture new root CID
new_root_cid = new_hamt.flush()
print("📦 new root CID:", new_root_cid)

# IPFS object size for apples-to-apples comparison
stat = kubo_cas._requests_session.post(
    kubo_cas.rpc_url, params={"arg": str(new_root_cid), "stat": True}
).json()
print("IPFS DAG size:", humanize.naturalsize(stat["CumulativeSize"]))


AttributeError: module 'zarr' has no attribute 'DirectoryStore'